In [1]:
import numpy as np
from collections import Counter 
import gensim.downloader as api # I used this function just to download my training dataset through api, the word2vec algorithm and core training loop is using only numpy

## Preprocessing data

In [2]:
class Preprocessor:
    def __init__(self, window_size, min_count=2): 
        self.window_size = window_size  
        self.min_count = min_count  # For rare case omiting
        self.word2id = {} 
        self.id2word = {}
        self.vocab_size = 0 
        self.neg_sampling_dist = [] 
        self.stop_words = {'the', 'a', 'an', 'and', 'or', 'but', 'in', 'on', 'at', 'to', 'for', 'of', 'with', 'is', 'was', 'were', 'it', 'as'} # Set of stop words to filter out

    def build_dict(self, words_list):
        print("Building a dictionary: ")
        
        # Claening data of stop words
        clean_words = [w for w in words_list if w not in self.stop_words and len(w) > 1]
        
        # Counting repetitions
        word_counts = Counter(clean_words) 
        valid_words = {w: c for w, c in word_counts.items() if c >= self.min_count}
        
        # Building dict 
        for word in valid_words.keys():
            if word not in self.word2id:
                self.word2id[word] = self.vocab_size
                self.id2word[self.vocab_size] = word
                self.vocab_size += 1 
        
        print(f"-Found {self.vocab_size} unique words ")

        # 3/4 trick
        if self.vocab_size > 0:
            total_words_pow = sum([count**0.75 for count in valid_words.values()])
            self.neg_sampling_dist = np.zeros(self.vocab_size)
            
            for word, count in valid_words.items():
                word_id = self.word2id[word]
                self.neg_sampling_dist[word_id] = (count**0.75) / total_words_pow
        
        return [w for w in clean_words if w in self.word2id]

    def generate_training_dataset(self, words_list):
        print(f"Generating training dataset: ")
        training_data = []
        
        word_ids = [self.word2id[w] for w in words_list if w in self.word2id]
        
        for i, center_word_id in enumerate(word_ids):
            start = max(0, i - self.window_size)
            end = min(len(word_ids), i + self.window_size + 1)
            
            for j in range(start, end):
                if i != j: 
                    context_word_id = word_ids[j]
                    training_data.append((center_word_id, context_word_id))
                    
        print(f"-Generated {len(training_data)} training pairs")
        return training_data

## Word2vec algorithm (skip gram negative sampling version)

In [3]:
class Word2Vec:
    def __init__(self, vocab_size, embedding_dim=100, learning_rate=0.01):
        self.n = embedding_dim
        self.v_size = vocab_size
        self.lr = learning_rate
        
        # Initializing matrixes of weights (U and V)

        self.V = np.random.randn(vocab_size, embedding_dim) * 0.1
        self.U = np.random.randn(vocab_size, embedding_dim) * 0.1

    def _sigmoid(self, x):
        return 1 / (1 + np.exp(-x))

    def forward(self, center_id, target_ids):
        # Central word vector
        self.vc = self.V[center_id] 
        
        # Candidates vector from U matrix
        self.u_candidates = self.U[target_ids]
        
        # Calculating z vector
        self.z = np.dot(self.u_candidates, self.vc)
        
        # Turning z vector into probabilites
        self.probs = self._sigmoid(self.z)
        
        return self.probs

    # Introducting an optimization objective
    def calculate_loss(self, probs):
        pos_loss = -np.log(probs[0] + 1e-10) # probs[0] is a true neighbor
        neg_loss = -np.sum(np.log(1 - probs[1:] + 1e-10)) # because sigma(-x) = 1 - sigma(x)

        return pos_loss + neg_loss
    
    # Backpropagation
    def backward(self, probs, target_ids):
        y_true = np.zeros_like(probs)
        y_true[0] = 1.0
        self.errors = probs - y_true # penalty for mistakes
        
        # Grads for matrix U 
        self.grad_U = np.outer(self.errors, self.vc)
        # Grad for central vector
        self.grad_vc = np.dot(self.errors, self.u_candidates)
        
        return self.grad_U, self.grad_vc
    
    # Parameter Updates via SGD
    def update_parameters(self, center_id, target_ids):

        self.U[target_ids] -= self.lr * self.grad_U
        self.V[center_id] -= self.lr * self.grad_vc

    # Using cosine Similarity to indicate the probability of neighboring
    def get_similarities(self, word, prep):

        if word not in prep.word2id: return None
        #Reference vector for our target word
        target_vec = self.V[prep.word2id[word]]
        sims = []
        for i in range(self.v_size):
            if i == prep.word2id[word]: continue
            v_i = self.V[i]
            # Standard Cosine Similarity formula: (A dot B) / (||A|| * ||B||)
            score = np.dot(target_vec, v_i) / (np.linalg.norm(target_vec) * np.linalg.norm(v_i) + 1e-10)
            sims.append((prep.id2word[i], score))
        return sims
    
    # Returns the top_n most similar words
    def most_similar(self, word, prep, top_n=5):
        sims = self.get_similarities(word, prep)
        if not sims: return "Word does not occur"
        return sorted(sims, key=lambda x: x[1], reverse=True)[:top_n]

    # Returns the top_n least similar words
    def least_similar(self, word, prep, top_n=5):
        sims = self.get_similarities(word, prep)
        if not sims: return "Word does not occur"
        return sorted(sims, key=lambda x: x[1])[:top_n] 

## Training process and small testing

In [4]:
# DATA PREPARATION
print("Downloading and cleaning the data: ")
dataset = api.load("text8") # The only time I used gensim library to speed up loading the data process


raw_tokens = []
for i, line in enumerate(dataset):
    raw_tokens.extend(line)
    if i > 20: break  

# Preprocessor Initialization
prep = Preprocessor(window_size=3, min_count=3)
clean_tokens = prep.build_dict(raw_tokens)
training_pairs = prep.generate_training_dataset(clean_tokens)

# Word2Vec Initialization
print("\nModel Initialization: ")
model = Word2Vec(vocab_size=prep.vocab_size, embedding_dim=100, learning_rate=0.03)
epochs = 10
num_negatives = 10

# Training Loop
print("\nStarting training process: ")

for epoch in range(epochs):
    total_loss = 0
    np.random.shuffle(training_pairs)
    
    for i in range(len(training_pairs)):
        center_id, context_id = training_pairs[i]
        
        # sampling Negative samples
        neg_ids = np.random.choice(prep.vocab_size, size=num_negatives, p=prep.neg_sampling_dist)
        target_ids = [context_id] + list(neg_ids)
        
        # Word2vec
        probs = model.forward(center_id, target_ids)
        loss = model.calculate_loss(probs)
        model.backward(probs, target_ids)
        model.update_parameters(center_id, target_ids)
        
        total_loss += loss
        
    print(f"   Epoch {epoch+1:2d}/{epochs} | Loss: {total_loss:.2f}")


# SMALL TEST
print("\nVECTOR SIMILARITY EVALUATION\n" + "="*40)

test_words = ['king', 'computer', 'science', 'god', 'american']

for word in test_words:
    if word in prep.word2id:
        print(f"\nWORD: **{word.upper()}**")
        
        top_matches = model.most_similar(word, prep, top_n=5)
        bottom_matches = model.least_similar(word, prep, top_n=5)
        
        print("   TOP SIMILAR: ")
        if isinstance(top_matches, list):
            for w, s in top_matches:
                print(f"      {s:.4f} | {w:<15} ")
        else: print(f"      {top_matches}")
            
        print("   LEAST SIMILAR: ")
        if isinstance(bottom_matches, list):
            for w, s in bottom_matches:
                print(f"      {s:.4f} | {w:<15}")
        else: print(f"      {bottom_matches}")
            
        print("-" * 30)
    else:
        print(f"\n WORD: {word.upper()} does not occur in our dict")

Building a dictionary: 
-Found 7486 unique words 
Generating training dataset: 
-Generated 875856 training pairs

Model Initialization: 

Starting training process: 
   Epoch  1/10 | Loss: 3183788.23
   Epoch  2/10 | Loss: 2588323.95
   Epoch  3/10 | Loss: 2446923.47
   Epoch  4/10 | Loss: 2332548.37
   Epoch  5/10 | Loss: 2240153.22
   Epoch  6/10 | Loss: 2166032.80
   Epoch  7/10 | Loss: 2106248.12
   Epoch  8/10 | Loss: 2057490.95
   Epoch  9/10 | Loss: 2018438.61
   Epoch 10/10 | Loss: 1986067.03

VECTOR SIMILARITY EVALUATION

WORD: **KING**
   TOP SIMILAR: 
      0.5572 | coronis         
      0.5067 | thessaly        
      0.5063 | cyparissus      
      0.4907 | priam           
      0.4876 | visiting        
   LEAST SIMILAR: 
      -0.2365 | oil            
      -0.1732 | acquired       
      -0.1283 | dhabi          
      -0.1265 | rise           
      -0.1217 | facilities     
------------------------------

WORD: **COMPUTER**
   TOP SIMILAR: 
      0.5406 | animator 